# Preprocessing

In [5]:
%load_ext autoreload
%autoreload 2
import os
from pathlib import Path
import joblib as jl
import numpy as np
import polars as pl
import pandas as pd
from utils import pipeline_v2, funcs
from utils.funcs import cooccurrence_matrix_generator, song_vector, save_glove_csv
from utils.pipeline_v2 import main_pipeline
from utils.GloVe import GloVe
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from collections import defaultdict

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [6]:
data_dir = Path("data")
data_dir.mkdir(parents=True, exist_ok=True)

subfolders = ["data-GloVe", "data-transformers", "data-RF", "data-w2v"]

for sub in subfolders:
    path = Path(data_dir) / sub
    path.mkdir(parents=True, exist_ok=True)

glove_dir=data_dir / "data-GloVe"
tran_dir=data_dir / "data-transformers"
rf_dir=data_dir / "data-RF"
w2v_dir=data_dir / "data-w2v"

## Reading and Splitting the Training Data

To optimize data preprocessing in the pipeline, we load the raw data into a **Polars DataFrame**, leveraging its multi-threaded engine to handle large files faster and more efficiently than pandas. 

To manage local memory constraints, the dataset is split into two equal partitions (`sample_df1` and `sample_df2`) and processed sequentially.

In [7]:
df_train = pl.read_csv(data_dir / 'train.csv')
df_test = pl.read_csv(data_dir / 'test.csv')

### Removing Duplicate Songs

Lyrics sites often contain the same song more than once (covers, remasters, re-uploads). If one copy ends up in the training split and another in the validation split, the model is rewarded for memorizing instead of generalizing. Exact duplicates are therefore removed from the training data **before** any cleaning or splitting. The number of test lyrics that also appear in the training data is printed for reference only; test rows are never dropped.

In [8]:
print("train rows before:", df_train.height)
print("duplicate lyrics in train:", df_train["lyrics"].is_duplicated().sum())
print("test lyrics also in train:", df_test["lyrics"].is_in(df_train["lyrics"].implode()).sum())

# keep the first copy of every song; the test set is left untouched because every id needs a prediction
df_train = df_train.unique(subset="lyrics", keep="first", maintain_order=True)
print("train rows after:", df_train.height)

train rows before: 134967
duplicate lyrics in train: 343
test lyrics also in train: 77
train rows after: 134789


In [9]:
sample_df1, sample_df2 = train_test_split(
    df_train,
    train_size=0.5,
    stratify=df_train['tag'],
    random_state=80085
)

### Lyrics Preprocessing Overview

Each partition is mapped through `main_pipeline` via Polars' `map_elements` to clean and standardize the `lyrics` column:

- **Text Cleaning & Normalization:** Strips song metadata tags (e.g., *[Chorus]*, *[Verse]*), URLs, brackets, and punctuation, while expanding contractions.
- **Filtering & Tokenization:** Tokenizes text into individual words, discards non-alphabetic tokens, and removes standard English stopwords as well as common musical filler vocalizations (e.g., *"oh"*, *"yeah"*, *"la"*).
- **Linguistic Standardization:** Converts diacritics to ASCII, applies lowercasing, and lemmatizes tokens across multiple parts of speech.
- **Output Schema:** With `tokenized_output=True`, each song's lyrics are transformed into a list of cleaned tokens (`pl.List(pl.String)`) stored under `cleaned_lyrics`.

Once both batches finish processing, they are combined back into a single unified dataset (`cleaned_df`) using vertical concatenation (`pl.concat`).

In [10]:
sample_df1 = sample_df1.with_columns(
    pl.col("lyrics")
    .map_elements(
        lambda x: main_pipeline(x, tokenized_output=True), #set to False for transformer
        return_dtype=pl.List(pl.String)#if tokenized_output=False this line should be removed
    )
    .alias("cleaned_lyrics")
)

sample_df2 = sample_df2.with_columns(
    pl.col("lyrics")
    .map_elements(
        lambda x: main_pipeline(x, tokenized_output=True), #set to False for transformer
        return_dtype=pl.List(pl.String) #if tokenized_output=False this line should be removed
    )
    .alias("cleaned_lyrics")
)

cleaned_df_train = pl.concat([sample_df1, sample_df2])

cleaned_df_test = df_test.with_columns(
    pl.col("lyrics")
    .map_elements(
        lambda x: main_pipeline(x, tokenized_output=True), #set to False for transformer
        return_dtype=pl.List(pl.String)#if tokenized_output=False this line should be removed
    )
    .alias("cleaned_lyrics"))

In [ ]:
sample_df1 = sample_df1.with_columns(
    pl.col("lyrics")
    .map_elements(
        lambda x: main_pipeline(x, tokenized_output=False)
    )
    .alias("cleaned_lyrics")
)

sample_df2 = sample_df2.with_columns(
    pl.col("lyrics")
    .map_elements(
        lambda x: main_pipeline(x, tokenized_output=False)
    )
    .alias("cleaned_lyrics")
)

cleaned_df_train_tran = pl.concat([sample_df1, sample_df2])

cleaned_df_test_tran = df_test.with_columns(
    pl.col("lyrics")
    .map_elements(
        lambda x: main_pipeline(x, tokenized_output=False)
    )
    .alias("cleaned_lyrics"))

## Checkpointing & Class Distribution Overview

To preserve our preprocessed data and avoid redundant recomputation, we export the DataFrame to **Parquet format** and reload it. Parquet provides efficient column-oriented compression and natively preserves complex nested schemas, such as columns containing lists of strings (`pl.List(pl.String)`).

An initial inspection of the target genre distribution reveals a notable class imbalance, with **Rock**, **Pop**, and **Rap** making up the substantial majority of the dataset. Downstream modeling and evaluation strategies will need to account for this skew.

In [ ]:
# transformers get plain cleaned text (tokens joined by spaces) and keep the id
cleaned_df_train_tran.select(['id', 'tag', pl.col('cleaned_lyrics').list.join(' ')]).write_parquet(tran_dir / 'cleaned_train_df.parquet')
cleaned_df_test_tran.select(['id', pl.col('cleaned_lyrics').list.join(' ')]).write_parquet(tran_dir / 'cleaned_test_df.parquet')

In [12]:
cleaned_df_train.select(['id', 'tag', 'cleaned_lyrics']).write_parquet(w2v_dir / 'cleaned_train_df_tokenized.parquet')
cleaned_df_test.select(['id', 'cleaned_lyrics']).write_parquet(w2v_dir / 'cleaned_test_df_tokenized.parquet')

In [13]:
cleaned_df_train = pl.read_parquet(w2v_dir / 'cleaned_train_df_tokenized.parquet')
cleaned_df_test = pl.read_parquet(w2v_dir / 'cleaned_test_df_tokenized.parquet')

In [14]:
cleaned_df_train['tag'].value_counts()

tag,count
str,u32
"""misc""",5620
"""rock""",25323
"""rap""",38582
"""country""",3463
"""pop""",55601
"""rb""",6200


## Train/Validation Split

The cleaned training songs are split 75/25, stratified by genre, so the validation set keeps the natural genre distribution. Everything that learns from data (co-occurrence matrix, GloVe, TF-IDF) is fitted on `X_train` only. The song ids of both parts are saved to `data/splits.parquet`, so every model (random forest, coevolution forest, transformer) can use exactly the same validation set.

In [15]:
X = cleaned_df_train.select('id', 'cleaned_lyrics')
y = cleaned_df_train.select('tag')

In [16]:
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.25, random_state=80085, stratify=y
)

In [17]:
# which song went where, in the same row order as the feature files
pl.concat([
    X_train.select("id").with_columns(pl.lit("train").alias("split")),
    X_val.select("id").with_columns(pl.lit("val").alias("split")),
]).write_parquet(data_dir / "splits.parquet")

## Dataset Branching: Co-Occurrence Matrix Generation

Rather than restricting downstream modeling to a single architecture, we branch our data preparation to support varied embedding strategies, ranging from traditional distributional methods (Word2Vec, GloVe) to deep sequence models (Transformers).

### Co-Occurrence Computation
Count-based embeddings require pairwise co-occurrence frequencies across the corpus. Computing a full co-occurrence matrix naively can trigger SciPy out-of-memory errors due to quadratic vocabulary growth. 

We run `cooccurrence_matrix_generator` to slice matrix operations into manageable chunks, prune self-co-occurrences, and persist the output as a compact Parquet file (`data/cooccurrence_matrix.parquet`).

In [18]:
df_cooccurrence = cooccurrence_matrix_generator(X_train, "cleaned_lyrics")
df_cooccurrence.write_parquet(glove_dir / "cooccurrence_matrix.parquet")

Computing pairs in blocks: 100%|██████████| 30/30 [00:34<00:00,  1.15s/it]


Assembling final co-occurrence table...


In [19]:
cooc_matrix = pl.scan_parquet(glove_dir / 'cooccurrence_matrix.parquet')
# how many distinct words? (cheap, works on the integer codes)
print(cooc_matrix.select(pl.col("word_1").n_unique()).collect(engine="streaming"))

# the 100k most frequent words
top = (cooc_matrix.group_by("word_1").agg(pl.col("count").sum())
         .sort("count", descending=True).head(100_000)
         .collect(engine="streaming")["word_1"].cast(pl.String))
vocab = top.sort()
enum = pl.Enum(vocab)

# filter, encode to ids, and write to disk, all in chunks
(cooc_matrix.filter(pl.col("count") > 1)                          # drops most of the noise first
   .with_columns(pl.col("word_1", "word_2").cast(pl.String))
   .filter(pl.col("word_1").is_in(vocab) & pl.col("word_2").is_in(vocab))
   .select(pl.col("word_1").cast(enum).to_physical().cast(pl.Int32).alias("i"),
           pl.col("word_2").cast(enum).to_physical().cast(pl.Int32).alias("j"),
           pl.col("count").cast(pl.Float32).alias("x"))
   .sink_parquet(glove_dir / "pairs.parquet"))

vocab.to_frame("word").write_parquet(glove_dir / "vocab.parquet")
print("pairs kept:", pl.scan_parquet(glove_dir / "pairs.parquet").select(pl.len()).collect())

<sys>:0: UserWarning: POLARS_MAX_BLOCKING_THREAD_COUNT reached (512), this may indicate a deadlock


shape: (1, 1)
┌────────┐
│ word_1 │
│ ---    │
│ u32    │
╞════════╡
│ 148286 │
└────────┘


C:\Users\Lucas\AppData\Local\Temp\ipykernel_28856\3803949885.py:19: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  .sink_parquet(glove_dir / "pairs.parquet"))


pairs kept: shape: (1, 1)
┌──────────┐
│ len      │
│ ---      │
│ u32      │
╞══════════╡
│ 91848144 │
└──────────┘


In [20]:
pairs = pl.read_parquet(glove_dir / "pairs.parquet")
i, j, x = pairs["i"].to_numpy(), pairs["j"].to_numpy(), pairs["x"].to_numpy()
del pairs

idx = np.arange(len(x), dtype=np.int32)
np.random.default_rng(0).shuffle(idx)
np.save(glove_dir / "i.npy", i[idx]); np.save(glove_dir / "j.npy", j[idx]); np.save(glove_dir / "x.npy", x[idx])
del i, j, x, idx

In [21]:
vocab = pl.read_parquet(glove_dir/"vocab.parquet")["word"]
glove = GloVe(vocab, dim=25).fit(glove_dir / "i.npy", glove_dir / "j.npy", glove_dir / "x.npy", epochs=30)
print(glove.most_similar("love"))
glove.save(glove_dir / "glove_25d")

c:\Users\Lucas\Github\Music-Genre-Identifier\utils\GloVe.py:73: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:209.)
  return torch.as_tensor(np.asarray(arr), dtype=dtype, device=self.device)


epoch 1/30  loss 0.1617
epoch 2/30  loss 0.0244
epoch 3/30  loss 0.0218
epoch 4/30  loss 0.0192
epoch 5/30  loss 0.0160
epoch 6/30  loss 0.0138
epoch 7/30  loss 0.0128
epoch 8/30  loss 0.0121
epoch 9/30  loss 0.0117
epoch 10/30  loss 0.0114
epoch 11/30  loss 0.0112
epoch 12/30  loss 0.0110
epoch 13/30  loss 0.0109
epoch 14/30  loss 0.0108
epoch 15/30  loss 0.0106
epoch 16/30  loss 0.0105
epoch 17/30  loss 0.0104
epoch 18/30  loss 0.0103
epoch 19/30  loss 0.0101
epoch 20/30  loss 0.0100
epoch 21/30  loss 0.0100
epoch 22/30  loss 0.0099
epoch 23/30  loss 0.0098
epoch 24/30  loss 0.0097
epoch 25/30  loss 0.0096
epoch 26/30  loss 0.0096
epoch 27/30  loss 0.0095
epoch 28/30  loss 0.0095
epoch 29/30  loss 0.0094
epoch 30/30  loss 0.0094
[('see', 0.993), ('come', 0.993), ('never', 0.992), ('way', 0.992), ('feel', 0.991), ('one', 0.99), ('let', 0.99), ('oh', 0.99)]


In [22]:
vectors, vocab = GloVe.load_vectors(glove_dir / "glove_25d")
index = {word: k for k, word in enumerate(vocab)}

In [23]:
X_train_glove = np.vstack([song_vector(t, vectors, index) for t in X_train['cleaned_lyrics']])
X_val_glove = np.vstack([song_vector(t, vectors, index) for t in X_val['cleaned_lyrics']])

In [24]:
train_df = save_glove_csv(X_train_glove, y_train, "data/data-GloVe/train_glove.csv")
val_df = save_glove_csv(X_val_glove, y_val, "data/data-GloVe/val_glove.csv")

print(train_df.shape)
print(train_df.show())

(101091, 26)


dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,dim_9,dim_10,dim_11,dim_12,dim_13,dim_14,dim_15,dim_16,dim_17,dim_18,dim_19,dim_20,dim_21,dim_22,dim_23,dim_24,genre
f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,str
0.667363,0.832688,-0.852087,0.973679,0.910192,-0.850839,-0.830453,0.964499,0.971202,0.848595,-0.573329,0.930607,0.78315,0.896505,0.895282,-0.632188,-0.815336,-1.011869,0.940717,0.906469,0.948654,-0.971175,0.728375,0.901237,-0.999605,"""rock"""
0.830466,0.811816,-0.779726,0.997481,1.040176,-0.925779,-1.056981,0.895307,0.978017,0.989439,-0.574792,1.16641,0.806457,0.88846,0.99357,-0.89902,-0.605206,-0.884007,1.043912,0.927609,0.973279,-1.090471,0.862568,1.012974,-1.024053,"""pop"""
1.290208,1.081013,-1.106177,0.952238,1.031012,-1.068913,-0.739278,1.07198,0.999081,1.136708,-0.798716,1.180604,0.702253,1.03814,1.115071,-1.130507,-0.856795,-0.784936,1.123778,1.053192,1.031829,-1.103493,0.594833,1.013553,-0.952891,"""pop"""
1.367819,0.954914,-0.95728,0.931842,0.938319,-1.08378,-0.991519,0.991043,0.992488,1.028137,-0.778894,1.176661,0.829521,0.982899,0.99818,-1.112118,-0.762383,-0.688467,1.059164,1.028718,0.9987,-1.001019,0.688743,0.935562,-0.882565,"""rap"""
1.12601,1.052042,-1.082255,0.961919,0.987967,-1.07866,-0.823669,1.000935,0.980929,1.089435,-0.753152,1.060501,0.759789,0.980748,1.05938,-1.090142,-0.834975,-0.757454,1.074985,0.993458,1.040745,-1.050225,0.659651,0.980013,-0.870548,"""pop"""


None


## TF-IDF Features and GloVe Combination

To add more information for the ML part, the averaged GloVe vectors are combined with TF-IDF word weights. The TF-IDF vocabulary is fit on the training split only and then applied unchanged to validation and test. Each TF-IDF dataframe is joined column-wise to its GloVe dataframe (same rows, same order), with the `genre` label as the last column, and saved to `data-RF`. Row order follows `data/splits.parquet` for train and validation, and `test.csv` for test.

In [25]:
tfidf = TfidfVectorizer(analyzer=lambda tokens: tokens, max_features=1000, sublinear_tf=True)
tfidf.fit(X_train["cleaned_lyrics"].to_list())
tfidf_cols = [f"tfidf_{w}" for w in tfidf.get_feature_names_out()]

def make_tfidf_df(tokens_col):
    X = tfidf.transform(tokens_col.to_list()).astype(np.float32)
    return pl.DataFrame(X.toarray(), schema=tfidf_cols, orient="row")

tfidf_train = make_tfidf_df(X_train["cleaned_lyrics"])
tfidf_val = make_tfidf_df(X_val["cleaned_lyrics"])
tfidf_test = make_tfidf_df(cleaned_df_test["cleaned_lyrics"])

In [26]:
train_glove = pl.read_csv(glove_dir / "train_glove.csv")
val_glove = pl.read_csv(glove_dir / "val_glove.csv")

X_test_glove = np.vstack([song_vector(t, vectors, index) for t in cleaned_df_test["cleaned_lyrics"]])
test_glove = pl.DataFrame(X_test_glove, schema=[f"dim_{k}" for k in range(X_test_glove.shape[1])], orient="row")

assert train_glove.height == tfidf_train.height, "train rows differ"
assert val_glove.height == tfidf_val.height, "val rows differ"
assert test_glove.height == tfidf_test.height, "test rows differ"

# GloVe columns, then TF-IDF columns, with the genre label last
train_rf = pl.concat([train_glove.drop("genre"), tfidf_train, train_glove.select("genre")], how="horizontal")
val_rf = pl.concat([val_glove.drop("genre"), tfidf_val, val_glove.select("genre")], how="horizontal")
test_rf = pl.concat([test_glove, tfidf_test], how="horizontal")

train_rf.write_parquet(rf_dir / "train_glove_tfidf.parquet")
val_rf.write_parquet(rf_dir / "val_glove_tfidf.parquet")
test_rf.write_parquet(rf_dir / "test_glove_tfidf.parquet")
print(train_rf.shape, val_rf.shape, test_rf.shape)

(101091, 1026) (33698, 1026) (33742, 1025)


In [27]:
train_rf.show()

dim_0,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,dim_9,dim_10,dim_11,dim_12,dim_13,dim_14,dim_15,dim_16,dim_17,dim_18,dim_19,dim_20,dim_21,dim_22,dim_23,dim_24,tfidf_across,tfidf_act,tfidf_action,tfidf_add,tfidf_afraid,tfidf_age,tfidf_ago,tfidf_ah,tfidf_ahead,tfidf_aim,tfidf_air,tfidf_alive,…,tfidf_win,tfidf_wind,tfidf_window,tfidf_wine,tfidf_wing,tfidf_winter,tfidf_wish,tfidf_wit,tfidf_within,tfidf_without,tfidf_woah,tfidf_woman,tfidf_wonder,tfidf_wood,tfidf_word,tfidf_work,tfidf_world,tfidf_worry,tfidf_worth,tfidf_would,tfidf_wrap,tfidf_wrist,tfidf_write,tfidf_wrong,tfidf_ya,tfidf_yah,tfidf_yea,tfidf_yeah,tfidf_year,tfidf_yes,tfidf_yesterday,tfidf_yet,tfidf_yo,tfidf_young,tfidf_yuh,tfidf_zone,genre
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,…,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,f32,str
0.6673626,0.8326884,-0.852087,0.9736793,0.9101918,-0.850839,-0.830453,0.9644992,0.971202,0.8485952,-0.573329,0.930607,0.7831501,0.896505,0.8952819,-0.632188,-0.815336,-1.011869,0.9407166,0.9064692,0.9486535,-0.971175,0.728375,0.901237,-0.999605,0.151381,0.0,0.0,0.0,0.0,0.155597,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.102484,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""rock"""
0.8304658,0.8118159,-0.779726,0.9974806,1.0401762,-0.925779,-1.056981,0.8953065,0.978017,0.9894387,-0.574792,1.1664099,0.806457,0.8884595,0.9935698,-0.899021,-0.605206,-0.884007,1.0439119,0.927609,0.973279,-1.090471,0.8625681,1.0129741,-1.024053,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.096758,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""pop"""
1.2902081,1.0810131,-1.106177,0.9522376,1.0310122,-1.068913,-0.739278,1.0719805,0.999081,1.1367079,-0.798716,1.1806045,0.702253,1.0381399,1.1150709,-1.130507,-0.856795,-0.784936,1.1237777,1.0531919,1.0318289,-1.103493,0.594833,1.0135534,-0.952891,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.083916,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.073271,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""pop"""
1.3678188,0.954914,-0.95728,0.9318417,0.9383186,-1.08378,-0.991519,0.9910428,0.9924877,1.028137,-0.778894,1.1766607,0.8295205,0.982899,0.99818,-1.112119,-0.762383,-0.688467,1.0591639,1.0287184,0.9986999,-1.001019,0.6887427,0.935562,-0.882565,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.116883,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""rap"""
1.1260102,1.0520416,-1.082254,0.9619191,0.9879669,-1.07866,-0.823669,1.0009351,0.9809288,1.0894353,-0.753152,1.0605015,0.7597891,0.980748,1.05938,-1.090142,-0.834975,-0.757454,1.074985,0.9934581,1.0407455,-1.050225,0.6596505,0.9800129,-0.870548,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,…,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.08283,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,"""pop"""
